# Source A — Grid Electricity Consumption (AETN 2024)
## Norte Amazónica Bolivia — Reality Demand Scenario

This notebook constructs **Source A** of the *Reality* demand scenario for 21 isolated municipalities in Norte Amazónica, Bolivia (Pando department, northern Beni, and Ixiamas in La Paz).

### What is Source A?

**Source A** quantifies electricity sold through the public grid in 2024, disaggregated by AETN tariff category:

| AETN category | Interpretation |
|---|---|
| Residential | Household consumption |
| General | Commercial and institutional services |
| Industrial | Industrial processes |
| Public Lighting | Street lighting |
| Other | Miscellaneous / unclassified |

This is one of three demand sources feeding the *Reality* scenario in EnergyScope:

| Source | Content | Data |
|---|---|---|
| **A (this notebook)** | Grid electricity by municipality and tariff category | AETN 2024 |
| B | Off-grid electricity (solar, generator) | RAMP Reality scenario |
| C | Unserved demand | no consume |

### Contrast with the Sufficiency scenario

The parallel *Sufficiency* scenario relies exclusively on RAMP-simulated demand — a bottom-up estimate of minimum viable energy access per appliance type, independent of existing infrastructure. The *Reality* scenario instead anchors on observed AETN sales, which capture current consumption patterns including commercial activity, industrial loads, and energy inefficiencies not modelled in RAMP.

### Three methodological groups

Because AETN data coverage is uneven across our 21 municipalities, the reconstruction proceeds in three tiers:

| Group | Municipalities | Approach |
|---|---|---|
| 1 | 8 | Direct AETN reading (± geographic correction and trend-based 2024 estimates) |
| 2 | 6 | ENDE Cobija aggregate split via tiered census-weighted method |
| 3 | 7 | No AETN data — estimated from comparable municipalities via census HH ratios |

**Authoritative source**: AETN, *Anuario Estadístico 2024* (Autoridad de Fiscalización de Electricidad y Tecnología Nuclear). Values represent MWh sold to final consumers during calendar year 2024.

In [1]:
import pandas as pd
import numpy as np
import os

OUTPUT_DIR = "output"
os.makedirs(OUTPUT_DIR, exist_ok=True)

In [2]:
def parse_mwh(val):
    """Convert AETN MWh string ('1 079,51') to float. Returns 0.0 for missing/dash."""
    s = str(val).strip().replace('\xa0', '').replace(' ', '').replace(',', '.')
    return 0.0 if s in ('', '-', 'nan') else float(s)


def parse_breakdown(field_str):
    """Parse '<br>'-separated AETN breakdown into a category→MWh dict."""
    if pd.isna(field_str) or str(field_str).strip() == '':
        return {}
    result = {}
    for part in str(field_str).split('<br>'):
        part = part.strip()
        if ':' not in part:
            continue
        key, val_str = part.split(':', 1)
        val_str = val_str.strip().replace('\xa0', '').replace(' ', '').replace(',', '.')
        try:
            result[key.strip()] = float(val_str)
        except (ValueError, AttributeError):
            result[key.strip()] = 0.0
    return result


def cats_from_bd(bd):
    return (
        bd.get('Residential',     0.0),
        bd.get('General',         0.0),
        bd.get('Industrial',      0.0),
        bd.get('Public Lighting', 0.0),
        bd.get('Other',           0.0),
    )


def make_result(municipality, department, total, c_tuple, method, comparable='', notes=''):
    res, gen, ind, pl, oth = c_tuple
    return {
        'municipality':         municipality,
        'department':           department,
        'total_MWh':            round(total, 2),
        'residential_MWh':      round(res,   2),
        'general_MWh':          round(gen,   2),
        'industrial_MWh':       round(ind,   2),
        'public_lighting_MWh':  round(pl,    2),
        'other_MWh':            round(oth,   2),
        'method':               method,
        'comparable_reference': comparable,
        'notes':                notes,
    }


## 1. Load AETN 2024 Data

The raw AETN file is a semicolon-delimited CSV (UTF-8-BOM) with 7 columns:

| Column | Description |
|---|---|
| Department | Bolivian department |
| Municipality / TIOC | Municipality or system name; `— 2024 estimate` suffix marks projected rows |
| System / Network | Utility system description (quoted; some fields contain internal semicolons) |
| Total Sales (MWh) | Annual electricity sales, formatted with space-thousands and comma-decimal |
| Sales Breakdown (MWh) | Per-category breakdown: `Residential: X<br>General: Y<br>...` |
| Total Subscribers | Total consumer count |
| Subscribers Breakdown | Per-category subscriber count |

Standard CSV parsing with `quotechar='"'` correctly handles the internal semicolons in quoted System/Network fields.

In [3]:
df_aetn = pd.read_csv(
    "data/electricity_beni_pando_aetn_2024.csv",
    sep=';',
    encoding='utf-8-sig',
    quotechar='"',
)
df_aetn.columns = [
    'dept', 'muni_raw', 'system',
    'total_mwh_raw', 'sales_bd_raw',
    'subscribers_raw', 'subs_bd_raw',
]


def get_aetn_row(muni_name):
    """Return (total_MWh, (res, gen, ind, pl, oth)) for a given AETN municipality_raw string."""
    mask = df_aetn['muni_raw'].str.strip() == muni_name
    rows = df_aetn[mask]
    if rows.empty:
        raise ValueError(f"AETN row not found: '{muni_name}'")
    r = rows.iloc[0]
    total = parse_mwh(r['total_mwh_raw'])
    bd = parse_breakdown(r['sales_bd_raw'])
    return total, cats_from_bd(bd)


print(f"AETN file loaded: {len(df_aetn)} rows")
df_aetn[['dept', 'muni_raw', 'total_mwh_raw']].dropna(subset=['total_mwh_raw'])

AETN file loaded: 19 rows


,dept,muni_raw,total_mwh_raw
1,Beni,Riberalta,"58 560,07"
2,Beni,Guayaramerín,"30 579,82"
3,Beni,Reyes,"4 489,55"
4,Beni,Santa Rosa,"4 142,59"
5,Beni,Exaltación,"264,12"
6,Beni,Exaltación — 2024 estimate,"311,40"
7,Beni,Rurrenabaque,"9 112,76"
8,Pando,Cobija,"70 571,36"
9,Pando,Puerto Gonzalo Moreno,"1 029,25"
10,Pando,Puerto Gonzalo Moreno — 2024 estimate,"1 079,51"


## 2. Group 1 — Direct AETN Data (8 municipalities)

Eight municipalities have individual entries in the AETN 2024 tables. For three of them (Exaltación, Puerto Gonzalo Moreno, Villa Nueva), the most recent complete individual year in the AETN is before 2024; the dataset provides trend-based `— 2024 estimate` rows to project values forward.

| Municipality | AETN entry used | Expected MWh |
|---|---|---|
| Riberalta | SA - ENDE Riberalta (2024) | 58 560.07 |
| Guayaramerín | SA - ENDE DELBENI Guayaramerín_TH (2024) minus Puerto Ustárez | ≈ 30 449.33 |
| Reyes | SIN - ENDE DELBENI (2024) | 4 489.55 |
| Santa Rosa (Beni) | SIN - ENDE DELBENI (2024) | 4 142.59 |
| Exaltación | 2024 estimate — trend 2018–2022 | 311.40 |
| El Sena | SA - ENDE (2024) | 3 690.32 |
| Puerto Gonzalo Moreno | 2024 estimate — trend 2020–2023 | 1 079.51 |
| Villa Nueva | 2024 estimate — trend 2020–2023 | 542.02 |

### Guayaramerín geographic correction

The AETN `Guayaramerín_TH` system (30 579.82 MWh) covers the municipality of Guayaramerín **plus** the hamlet of Puerto Ustárez, which belongs to a different administrative area. Puerto Ustárez must be subtracted; Cachuela Esperanza and Rosario del Yata are within the municipality and are retained.

The correction is applied proportionally across all tariff categories:
$$\text{corrected}_{\text{cat}} = \text{raw}_{\text{cat}} \times \frac{30\,449.33}{30\,579.82}$$

### Why use 2024 estimates for Exaltación, Puerto Gonzalo Moreno and Villa Nueva?

These three municipalities were absorbed into consolidated AETN system aggregates before 2024 (Exaltación merged into San Ignacio de Moxos in 2023; Puerto Gonzalo Moreno and Villa Nueva merged into the ENDE Pando consolidated account). The AETN dataset provides dedicated `— 2024 estimate` rows, extrapolated from the last available individual-year series, which are more accurate than re-using the last observed value directly.

In [4]:
results = []

# ── 2.1  Riberalta ─────────────────────────────────────────────────────────
total, c = get_aetn_row('Riberalta')
assert abs(total - 58560.07) < 0.1, f"Riberalta mismatch: {total}"
results.append(make_result('Riberalta', 'Beni', total, c,
    method='AETN_direct', notes='SA - ENDE Riberalta (2024)'))
print(f"Riberalta              : {total:>10.2f} MWh  ✓")

# ── 2.2  Guayaramerín — with Puerto Ustárez correction ────────────────────
total_raw, c_raw = get_aetn_row('Guayaramerín')
assert abs(total_raw - 30579.82) < 0.1, f"Guayaramerín raw mismatch: {total_raw}"

ustarez_total, _ = get_aetn_row('Puerto Ustárez — 2024 estimate')
assert abs(ustarez_total - 130.49) < 0.1, f"Puerto Ustárez mismatch: {ustarez_total}"

corrected_total = total_raw - ustarez_total       # = 30 449.33 MWh
factor = corrected_total / total_raw
c_corrected = tuple(v * factor for v in c_raw)

assert abs(corrected_total - 30449.33) < 0.1, f"Guayaramerín corrected mismatch: {corrected_total}"
results.append(make_result('Guayaramerín', 'Beni', corrected_total, c_corrected,
    method='AETN_minus_Ustarez',
    notes=(
        f'Guayaramerín_TH {total_raw:.2f} MWh − Puerto Ustárez {ustarez_total:.2f} MWh; '
        f'correction factor = {factor:.6f} applied to all categories'
    )))
print(f"Guayaramerín (raw)     : {total_raw:>10.2f} MWh")
print(f"  − Puerto Ustárez     :    −{ustarez_total:.2f} MWh")
print(f"  Corrected total      : {corrected_total:>10.2f} MWh  ✓  (factor = {factor:.6f})")

# ── 2.3  Reyes ─────────────────────────────────────────────────────────────
total, c = get_aetn_row('Reyes')
assert abs(total - 4489.55) < 0.1, f"Reyes mismatch: {total}"
results.append(make_result('Reyes', 'Beni', total, c,
    method='AETN_direct', notes='SIN - ENDE DELBENI (2024)'))
print(f"Reyes                  : {total:>10.2f} MWh  ✓")

# ── 2.4  Santa Rosa (Beni) ─────────────────────────────────────────────────
total, c = get_aetn_row('Santa Rosa')
assert abs(total - 4142.59) < 0.1, f"Santa Rosa (Beni) mismatch: {total}"
results.append(make_result('Santa Rosa', 'Beni', total, c,
    method='AETN_direct', notes='SIN - ENDE DELBENI (2024)'))
print(f"Santa Rosa (Beni)      : {total:>10.2f} MWh  ✓")

# ── 2.5  Exaltación — 2024 estimate ────────────────────────────────────────
total, c = get_aetn_row('Exaltación — 2024 estimate')
assert abs(total - 311.40) < 0.1, f"Exaltación 2024 est mismatch: {total}"
results.append(make_result('Exaltación', 'Beni', total, c,
    method='AETN_trend_estimate',
    notes='2024 estimate based on 2018–2022 trend (2023 excluded: partial year post-merger)'))
print(f"Exaltación (2024 est)  : {total:>10.2f} MWh  ✓")

# ── 2.6  El Sena ────────────────────────────────────────────────────────────
total, c = get_aetn_row('El Sena')
assert abs(total - 3690.32) < 0.1, f"El Sena mismatch: {total}"
results.append(make_result('El Sena', 'Pando', total, c,
    method='AETN_direct', notes='SA - ENDE (2024)'))
print(f"El Sena                : {total:>10.2f} MWh  ✓")

# ── 2.7  Puerto Gonzalo Moreno — 2024 estimate ─────────────────────────────
total, c = get_aetn_row('Puerto Gonzalo Moreno — 2024 estimate')
assert abs(total - 1079.51) < 0.1, f"Puerto Gonzalo Moreno 2024 est mismatch: {total}"
results.append(make_result('Puerto Gonzalo Moreno', 'Pando', total, c,
    method='AETN_trend_estimate',
    notes='2024 estimate based on 2020–2023 trend (2019 excluded as start-up outlier)'))
print(f"Puerto Gonzalo Moreno  : {total:>10.2f} MWh  ✓")

# ── 2.8  Villa Nueva — 2024 estimate ───────────────────────────────────────
total, c = get_aetn_row('Villa Nueva — 2024 estimate')
assert abs(total - 542.02) < 0.1, f"Villa Nueva 2024 est mismatch: {total}"
results.append(make_result('Villa Nueva', 'Pando', total, c,
    method='AETN_trend_estimate',
    notes='2024 estimate based on 2020–2023 trend (2019 excluded as start-up outlier)'))
print(f"Villa Nueva (2024 est) : {total:>10.2f} MWh  ✓")

group1_total = sum(r['total_MWh'] for r in results)
print(f"\nGroup 1 subtotal       : {group1_total:>10.2f} MWh")

Riberalta              :   58560.07 MWh  ✓
Guayaramerín (raw)     :   30579.82 MWh
  − Puerto Ustárez     :    −130.49 MWh
  Corrected total      :   30449.33 MWh  ✓  (factor = 0.995733)
Reyes                  :    4489.55 MWh  ✓
Santa Rosa (Beni)      :    4142.59 MWh  ✓
Exaltación (2024 est)  :     311.40 MWh  ✓
El Sena                :    3690.32 MWh  ✓
Puerto Gonzalo Moreno  :    1079.51 MWh  ✓
Villa Nueva (2024 est) :     542.02 MWh  ✓

Group 1 subtotal       :  103264.79 MWh


## 3. Group 2 — ENDE Cobija System Split (6 municipalities)

Six municipalities in Pando (Cobija, Porvenir, Bolpebra, Bella Flor, Filadelfia, Puerto Rico) are served by the ENDE Cobija consolidated system. AETN reports only a single aggregate of **70 571.36 MWh** with no individual municipal breakdown.

### Tiered split method

A naive proportional split by household count would be biased because per-household consumption differs systematically across municipalities — urban Cobija has commercial and institutional loads far exceeding those of small rural towns. Instead, each municipality is assigned a *reference rate* (MWh per grid-connected household) derived from a comparable municipality for which individual AETN data is available:

| Municipality | Comparable reference | Rationale |
|---|---|---|
| Cobija | Riberalta | Both are departmental capitals with mixed residential + commercial + institutional demand |
| Porvenir | Reyes | Both are mid-sized Amazonian district capitals |
| Puerto Rico | Santa Rosa (Beni) | Similar population size and economic structure |
| Filadelfia | Santa Rosa (Beni) | Similar population size and economic structure |
| Bella Flor | Exaltación 2024 | Both are small, predominantly rural |
| Bolpebra | Exaltación 2024 | Both are small, predominantly rural |

**Step 1** — Raw unscaled estimate for each municipality:
$$\text{raw\_MWh}(m) = HH_{\text{censo}}(m) \times \frac{\text{MWh}_{\text{ref}}}{HH_{\text{censo,ref}}}$$

**Step 2** — Scale so that the 6 municipalities sum to the known ENDE Cobija total:
$$\text{factor} = \frac{70\,571.36}{\sum_m \text{raw\_MWh}(m)}, \qquad \text{final\_MWh}(m) = \text{raw\_MWh}(m) \times \text{factor}$$

**Step 3** — Apply the ENDE Cobija system-wide tariff category proportions (from the AETN Cobija entry) to each municipality's final MWh.

> **Limitation**: the tiered approach reduces but does not eliminate per-household consumption bias. Cobija's unusually large commercial and public-sector base may still be underrepresented relative to Riberalta. Additionally, Bella Flor and Bolpebra are small enough (< 570 HH) that the proportional estimate carries substantial uncertainty.

In [5]:
df_census = pd.read_csv("output/CSV_final.csv")

COL_GRID = 'NÚMERO DE VIVIENDAS POR FUENTE DE ELECTRICIDAD | 2024 | Servicio público de energía eléctrica'


def get_hh(muni_tioc, dept):
    mask = (df_census['MUNICIPIO/TIOC'].str.strip() == muni_tioc) & \
           (df_census['DEPARTAMENTO'].str.strip()   == dept)
    val = df_census.loc[mask, COL_GRID]
    if val.empty:
        raise KeyError(f"Census lookup failed: '{muni_tioc}' ({dept})")
    return int(val.iloc[0])


HH_RIBERALTA  = get_hh('Riberalta',  'Beni')
HH_REYES      = get_hh('Reyes',      'Beni')
HH_SANTA_ROSA = get_hh('Santa Rosa', 'Beni')
HH_EXALTACION = get_hh('Exaltación', 'Beni')

print(f"Riberalta        : {HH_RIBERALTA:>6}  (expected 23 121)")
print(f"Reyes            : {HH_REYES:>6}  (expected  2 353)")
print(f"Santa Rosa (Beni): {HH_SANTA_ROSA:>6}  (expected  1 728)")
print(f"Exaltación       : {HH_EXALTACION:>6}  (expected    292)")

assert HH_RIBERALTA  == 23121
assert HH_REYES      ==  2353
assert HH_SANTA_ROSA ==  1728
assert HH_EXALTACION ==   292
print("All reference HH values verified ✓")


Riberalta        :  23121  (expected 23 121)
Reyes            :   2353  (expected  2 353)
Santa Rosa (Beni):   1728  (expected  1 728)
Exaltación       :    292  (expected    292)
All reference HH values verified ✓


In [6]:
MWH_RIBERALTA  = next(r['total_MWh'] for r in results if r['municipality'] == 'Riberalta')
MWH_REYES      = next(r['total_MWh'] for r in results if r['municipality'] == 'Reyes')
MWH_SANTA_ROSA = next(r['total_MWh'] for r in results if r['municipality'] == 'Santa Rosa')
MWH_EXALTACION = next(r['total_MWh'] for r in results if r['municipality'] == 'Exaltación')

RATE_RIBERALTA  = MWH_RIBERALTA  / HH_RIBERALTA
RATE_REYES      = MWH_REYES      / HH_REYES
RATE_SANTA_ROSA = MWH_SANTA_ROSA / HH_SANTA_ROSA
RATE_EXALTACION = MWH_EXALTACION / HH_EXALTACION

print("Reference rates (MWh per grid-connected HH):")
print(f"  Riberalta  : {RATE_RIBERALTA:.4f} MWh/HH")
print(f"  Reyes      : {RATE_REYES:.4f} MWh/HH")
print(f"  Santa Rosa : {RATE_SANTA_ROSA:.4f} MWh/HH")
print(f"  Exaltación : {RATE_EXALTACION:.4f} MWh/HH")

COBIJA_SYSTEM_MWH = 70571.36
cobija_total_check, cobija_cats = get_aetn_row('Cobija')
assert abs(cobija_total_check - COBIJA_SYSTEM_MWH) < 0.1
cobija_props = tuple(v / COBIJA_SYSTEM_MWH for v in cobija_cats)

# (municipality, dept, rate, comparable_label)
G2_MUNIS = [
    ('Cobija',      'Pando', RATE_RIBERALTA,  'Riberalta'),
    ('Porvenir',    'Pando', RATE_REYES,      'Reyes'),
    ('Puerto Rico', 'Pando', RATE_SANTA_ROSA, 'Santa Rosa'),
    ('Filadelfia',  'Pando', RATE_SANTA_ROSA, 'Santa Rosa'),
    ('Bella Flor',  'Pando', RATE_EXALTACION, 'Exaltación'),
    ('Bolpebra',    'Pando', RATE_EXALTACION, 'Exaltación'),
]

raw = {muni: get_hh(muni, dept) * rate for muni, dept, rate, _ in G2_MUNIS}
scale = COBIJA_SYSTEM_MWH / sum(raw.values())
print(f"\nCobija system: {COBIJA_SYSTEM_MWH} MWh  →  scaling factor: {scale:.6f}")

g2_results = []
for muni, dept, rate, comp in G2_MUNIS:
    mwh = raw[muni] * scale
    g2_results.append(make_result(
        muni, dept, mwh, tuple(p * mwh for p in cobija_props),
        method='ENDE_Cobija_split', comparable=comp,
        notes=f'ENDE Cobija {COBIJA_SYSTEM_MWH} MWh split; scale={scale:.6f}',
    ))
    print(f"  {muni:<12}: {mwh:.1f} MWh")

assert abs(sum(r['total_MWh'] for r in g2_results) - COBIJA_SYSTEM_MWH) < 0.15, "G2 sum mismatch"
print("Group 2 sum verified ✓")
results.extend(g2_results)


Reference rates (MWh per grid-connected HH):
  Riberalta  : 2.5328 MWh/HH
  Reyes      : 1.9080 MWh/HH
  Santa Rosa : 2.3973 MWh/HH
  Exaltación : 1.0664 MWh/HH

Cobija system: 70571.36 MWh  →  scaling factor: 1.459102
  Cobija      : 55141.5 MWh
  Porvenir    : 5011.2 MWh
  Puerto Rico : 4701.2 MWh
  Filadelfia  : 4589.3 MWh
  Bella Flor  : 885.4 MWh
  Bolpebra    : 242.7 MWh
Group 2 sum verified ✓


## 4. Group 3 — Estimations from Comparable Municipalities (7 municipalities)

Seven municipalities have no individual AETN entry and are outside the ENDE Cobija aggregate. Their electricity consumption is estimated by scaling from the closest comparable municipality using census household counts:

$$\text{estimated\_MWh}(m) = HH_{\text{censo}}(m) \times \frac{\text{MWh}_{\text{comparable}}}{HH_{\text{censo,comparable}}}$$

The comparable municipality's tariff category proportions are then applied to distribute the estimated total across the five output categories.

| Municipality | Census HH | Comparable | Comparable MWh | Comparable HH | Rate (MWh/HH) |
|---|---|---|---|---|---|
| Ixiamas | 1 470 | Reyes | 4 489.55 | 2 353 | 1.908 |
| San Lorenzo | 1 288 | Riberalta | 58 560.07 | 23 121 | 2.533 |
| Santa Rosa del Abuná | 568 | Exaltación 2024 | 311.40 | 292 | 1.066 |
| Santos Mercado | 188 | Exaltación 2024 | 311.40 | 292 | 1.066 |
| Nueva Esperanza | 71 | Exaltación 2024 | 311.40 | 292 | 1.066 |
| Ingavi | 113 | Exaltación 2024 | 311.40 | 292 | 1.066 |
| San Pedro | 2 | Exaltación 2024 | 311.40 | 292 | 1.066 |

**Comparable choice rationale**: Ixiamas (La Paz) and San Lorenzo (Pando) are medium-sized Amazonian towns most comparable in size and economic profile to Reyes. The remaining five municipalities are small, predominantly rural settlements analogous to Exaltación before its integration into the SIN network.

> **Note on San Lorenzo**: this municipality has no individual AETN entry and is not listed in the ENDE Cobija system description in the AETN 2024. It is therefore treated as Group 3. Reyes is chosen as comparable given similar population size (1 288 vs 2 353 grid-connected HH).

> **Limitation**: San Pedro (2 HH connected to the grid) and Nueva Esperanza (71 HH) are very small; their estimates are approximate and sensitive to the per-HH rate assumption.

In [7]:
_, reyes_cats = get_aetn_row('Reyes')
reyes_props = tuple(v / MWH_REYES for v in reyes_cats)

_, exalt_cats = get_aetn_row('Exaltación — 2024 estimate')
exalt_props = tuple(v / MWH_EXALTACION for v in exalt_cats)

# (municipality, dept, census_muni, rate, comparable, cat_props)
G3_MUNIS = [
    ('Ixiamas',              'La Paz', 'Ixiamas',         RATE_REYES,      'Reyes',      reyes_props),
    ('San Lorenzo',          'Pando',  'San Lorenzo',     RATE_REYES,      'Riberalta',  reyes_props),
    ('Santa Rosa del Abuná', 'Pando',  'Santa Rosa',      RATE_EXALTACION, 'Exaltación', exalt_props),
    ('Santos Mercado',       'Pando',  'Santos Mercado',  RATE_EXALTACION, 'Exaltación', exalt_props),
    ('Nueva Esperanza',      'Pando',  'Nueva Esperanza', RATE_EXALTACION, 'Exaltación', exalt_props),
    ('Ingavi',               'Pando',  'Ingavi',          RATE_EXALTACION, 'Exaltación', exalt_props),
    ('San Pedro',            'Pando',  'San Pedro',       RATE_EXALTACION, 'Exaltación', exalt_props),
]

g3_results = []
print("Group 3 — estimated consumption:")
for muni, dept, census_muni, rate, comp, props in G3_MUNIS:
    hh  = get_hh(census_muni, dept)
    mwh = hh * rate
    g3_results.append(make_result(
        muni, dept, mwh, tuple(p * mwh for p in props),
        method='Estimation', comparable=comp,
        notes=f'{hh} HH × {rate:.4f} MWh/HH (rate from {comp})',
    ))
    print(f"  {muni:<25}: {hh} HH × {rate:.4f} = {mwh:.1f} MWh")

results.extend(g3_results)


Group 3 — estimated consumption:
  Ixiamas                  : 1470 HH × 1.9080 = 2804.8 MWh
  San Lorenzo              : 1288 HH × 1.9080 = 2457.5 MWh
  Santa Rosa del Abuná     : 568 HH × 1.0664 = 605.7 MWh
  Santos Mercado           : 188 HH × 1.0664 = 200.5 MWh
  Nueva Esperanza          : 71 HH × 1.0664 = 75.7 MWh
  Ingavi                   : 113 HH × 1.0664 = 120.5 MWh
  San Pedro                : 2 HH × 1.0664 = 2.1 MWh


## 5. Output — Combined Dataset

All 21 municipalities are now combined into a single DataFrame. The output CSV `source_A_grid_consumption_by_municipality.csv` contains one row per municipality with columns:

| Column | Description |
|---|---|
| municipality | Municipality name |
| department | Bolivian department |
| total_MWh | Total grid electricity consumption (MWh) |
| residential_MWh | Residential tariff category (MWh) |
| general_MWh | General/commercial tariff category (MWh) |
| industrial_MWh | Industrial tariff category (MWh) |
| public_lighting_MWh | Public lighting tariff category (MWh) |
| other_MWh | Other / unclassified (MWh) |
| method | One of: `AETN_direct`, `AETN_minus_Ustarez`, `AETN_trend_estimate`, `ENDE_Cobija_split`, `Estimation` |
| comparable_reference | Comparable municipality used (Groups 2 and 3) |
| notes | Source description and key parameters |

In [8]:
df_out = pd.DataFrame(results)[[
    'municipality', 'department', 'total_MWh',
    'residential_MWh', 'general_MWh', 'industrial_MWh',
    'public_lighting_MWh', 'other_MWh',
    'method', 'comparable_reference', 'notes',
]]

out_path = os.path.join(OUTPUT_DIR, 'source_A_grid_consumption_by_municipality.csv')
df_out.to_csv(out_path, index=False, encoding='utf-8-sig')
print(f"Saved: {out_path}")
print(f"Rows : {len(df_out)} (expected 21)")
assert len(df_out) == 21, f"Expected 21 municipalities, got {len(df_out)}"

df_out

Saved: output\source_A_grid_consumption_by_municipality.csv
Rows : 21 (expected 21)


,municipality,department,total_MWh,residential_MWh,general_MWh,industrial_MWh,public_lighting_MWh,other_MWh,method,comparable_reference,notes
0,Riberalta,Beni,58560.07,31096.08,17292.26,5195.21,3987.20,989.32,AETN_direct,,SA - ENDE Riberalta (2024)
1,Guayaramerín,Beni,30449.33,18367.61,8547.77,2228.95,1243.08,61.93,AETN_minus_Ustarez,,Guayaramerín_TH 30579.82 MWh − Puerto Ustárez ...
2,Reyes,Beni,4489.55,2810.71,936.10,236.92,474.05,31.78,AETN_direct,,SIN - ENDE DELBENI (2024)
3,Santa Rosa,Beni,4142.59,2594.56,840.77,366.60,224.77,115.89,AETN_direct,,SIN - ENDE DELBENI (2024)
4,Exaltación,Beni,311.40,194.34,74.63,0.15,20.02,22.26,AETN_trend_estimate,,2024 estimate based on 2018–2022 trend (2023 e...
5,El Sena,Pando,3690.32,2076.35,1072.90,357.81,153.01,30.25,AETN_direct,,SA - ENDE (2024)
6,Puerto Gonzalo Moreno,Pando,1079.51,541.83,413.09,0.00,124.58,0.00,AETN_trend_estimate,,2024 estimate based on 2020–2023 trend (2019 e...
7,Villa Nueva,Pando,542.02,279.24,184.58,39.68,37.13,1.40,AETN_trend_estimate,,2024 estimate based on 2020–2023 trend (2019 e...
8,Cobija,Pando,55141.50,29903.61,18859.51,3861.17,2207.66,309.56,ENDE_Cobija_split,Riberalta,ENDE Cobija 70571.36 MWh split; scale=1.459102
9,Porvenir,Pando,5011.17,2717.59,1713.92,350.90,200.63,28.13,ENDE_Cobija_split,Reyes,ENDE Cobija 70571.36 MWh split; scale=1.459102


In [9]:
SEP = '=' * 70
print(SEP)
print('SUMMARY — Source A: Grid Electricity Consumption by Municipality')
print(SEP)

GROUP_ORDER = [
    ('AETN_direct',         'Group 1 — AETN direct (2024 data)'),
    ('AETN_minus_Ustarez',  'Group 1 — AETN direct (Guayaramerín corrected)'),
    ('AETN_trend_estimate', 'Group 1 — AETN trend estimate (2024 projection)'),
    ('ENDE_Cobija_split',   'Group 2 — ENDE Cobija aggregate split'),
    ('Estimation',          'Group 3 — Estimation from comparable'),
]

grand_total = 0.0
for method, label in GROUP_ORDER:
    subset = df_out[df_out['method'] == method]
    if subset.empty:
        continue
    subtotal = subset['total_MWh'].sum()
    grand_total += subtotal
    print(f'\n{label}:')
    for _, row in subset.iterrows():
        print(f"  {row['municipality']:<28}: {row['total_MWh']:>9.2f} MWh")
    print(f"  {'  Subtotal':<28}: {subtotal:>9.2f} MWh")

print('\n' + '-' * 70)
print(f"  {'GRAND TOTAL (21 municipalities)':<28}: {grand_total:>9.2f} MWh")
print('-' * 70)
print('\nCategory totals (all municipalities):')
for col, lbl in [
    ('residential_MWh',     'Residential'),
    ('general_MWh',         'General'),
    ('industrial_MWh',      'Industrial'),
    ('public_lighting_MWh', 'Public Lighting'),
    ('other_MWh',           'Other'),
]:
    val = df_out[col].sum()
    print(f"  {lbl:<20}: {val:>9.2f} MWh  ({val/grand_total*100:.1f}%)")


SUMMARY — Source A: Grid Electricity Consumption by Municipality

Group 1 — AETN direct (2024 data):
  Riberalta                   :  58560.07 MWh
  Reyes                       :   4489.55 MWh
  Santa Rosa                  :   4142.59 MWh
  El Sena                     :   3690.32 MWh
    Subtotal                  :  70882.53 MWh

Group 1 — AETN direct (Guayaramerín corrected):
  Guayaramerín                :  30449.33 MWh
    Subtotal                  :  30449.33 MWh

Group 1 — AETN trend estimate (2024 projection):
  Exaltación                  :    311.40 MWh
  Puerto Gonzalo Moreno       :   1079.51 MWh
  Villa Nueva                 :    542.02 MWh
    Subtotal                  :   1932.93 MWh

Group 2 — ENDE Cobija aggregate split:
  Cobija                      :  55141.50 MWh
  Porvenir                    :   5011.17 MWh
  Puerto Rico                 :   4701.25 MWh
  Filadelfia                  :   4589.31 MWh
  Bella Flor                  :    885.39 MWh
  Bolpebra              

---

## Part 2 — Residential End-Use Disaggregation

The residential AETN total for each municipality (from Part 1) is split into five EnergyScope end-uses:

| End-use | Appliances included |
|---|---|
| FOOD_PRESERVATION | Refrigerador |
| SPACE_COOLING | Aire acondicionado + ventilateur |
| HEAT_LOW_T_HW | Calefón/termotanque eléctrique |
| LIGHTING_R_C | Éclairage intérieur (indoor only) |
| ELECTRICITY | TV, radio, téléphone, laptop, blender, lavadora, antenne, microondas |

> **Outdoor lighting is excluded** from this residential breakdown. It will be combined with the *Public Lighting* tariff category in a later step.

### Method

The AETN residential total is **fixed** (measured). A bottom-up model computes the *proportions* between end-uses from appliance penetrations × annual kWh per appliance. A **uniform calibration factor** then scales each end-use so the sum exactly matches the AETN value:

```
bottom_up_EU  = Σ_appliances( kWh_appliance × penetration × HH_grid )
k             = residential_AETN_kWh / Σ(bottom_up_EU)
final_EU      = bottom_up_EU × k          ← sums to AETN exact
```

The calibration absorbs uncounted appliances and local usage deviations. It is applied uniformly, so the **relative shares** are entirely determined by the bottom-up model.

### Structure additionnable avec Source B

Les end-uses obtenus ici (source A, ménages connectés au réseau) sont structurés pour pouvoir être additionnés directement avec la source B (ménages hors réseau, modélisés par RAMP). Les colonnes correspondent aux mêmes end-uses EnergyScope.


### 2.1 Census Appliance Penetrations

Appliance ownership rates are read from two census 2024 tables:

- **EQUIPAMIENTO DEL HOGAR (sheet 2)**: refrigerador, microondas, calefón/termotanque, aire acondicionado, lavadora de ropa
- **TIC (sheet 1)**: televisor, radio/equipo de sonido, computadora/laptop/tablet, teléfono celular

**Penetration formula**: `p = Tiene / (Tiene + No_tiene)`.  
"Sin especificar" (non-response) households are **excluded** from the denominator: non-respondents are not counted as "does not own the appliance". Instead we assume that non-responses are distributed in the same proportion as known responses — a standard missing-data imputation hypothesis. This penetration is reused as-is in Source B, applied to `HH_nongrid` instead of `HH_grid`.

**Antenna device**: uses the penetration of the **radio** (same column as the radio appliance). An antenna captures radio/TV broadcast signals — this is physically coherent and avoids aberrant values that arise with Internet fijo in small municipalities.

**Internet fijo** (fixed internet access) is **excluded** from the appliance list: the census variable is unreliable in rural Norte Amazónica (high non-response, measurement inconsistencies in small municipalities).

**Name mapping**: the census uses "Sena" (not "El Sena") and a single "Santa Rosa" for Pando (disambiguated from Beni's Santa Rosa by the department filter).

In [10]:

CENSUS_NAME_MAP = {
    'El Sena':              'Sena',
    'Santa Rosa del Abuná': 'Santa Rosa',    # disambiguated by dept=Pando
}

def census_name(muni):
    return CENSUS_NAME_MAP.get(muni, muni)

# ── Equipment columns — Tiene ───────────────────────────────────────────────
CE_TOT    = 'NÚMERO DE HOGARES CON EQUIPAMIENTO DEL HOGAR | 2024 | Total hogares'
CE_FRG    = 'NÚMERO DE HOGARES CON EQUIPAMIENTO DEL HOGAR | 2024 | Refrigerador o congelador | Tiene'
CE_MCR    = 'NÚMERO DE HOGARES CON EQUIPAMIENTO DEL HOGAR | 2024 | Microondas | Tiene'
CE_CAL    = 'NÚMERO DE HOGARES CON EQUIPAMIENTO DEL HOGAR | 2024 | Calefón o termotanque | Tiene'
CE_AC     = 'NÚMERO DE HOGARES CON EQUIPAMIENTO DEL HOGAR | 2024 | Aire Acondicionado | Tiene'
CE_WAS    = 'NÚMERO DE HOGARES CON EQUIPAMIENTO DEL HOGAR | 2024 | Lavadora de ropa | Tiene'

# ── Equipment columns — No tiene ────────────────────────────────────────────
CE_FRG_NO = 'NÚMERO DE HOGARES CON EQUIPAMIENTO DEL HOGAR | 2024 | Refrigerador o congelador | No tiene'
CE_MCR_NO = 'NÚMERO DE HOGARES CON EQUIPAMIENTO DEL HOGAR | 2024 | Microondas | No tiene'
CE_CAL_NO = 'NÚMERO DE HOGARES CON EQUIPAMIENTO DEL HOGAR | 2024 | Calefón o termotanque | No tiene'
CE_AC_NO  = 'NÚMERO DE HOGARES CON EQUIPAMIENTO DEL HOGAR | 2024 | Aire Acondicionado | No tiene'
CE_WAS_NO = 'NÚMERO DE HOGARES CON EQUIPAMIENTO DEL HOGAR | 2024 | Lavadora de ropa | No tiene'

# ── TIC columns — Tiene ──────────────────────────────────────────────────────
CT_TOT    = 'NÚMERO DE HOGARES CON TECNOLOGÍAS TIC | 2024 | Total'
CT_TV     = 'NÚMERO DE HOGARES CON TECNOLOGÍAS TIC | 2024 | Televisor | Tiene'
CT_RAD    = 'NÚMERO DE HOGARES CON TECNOLOGÍAS TIC | 2024 | Radio o equipo de sonido | Tiene'
CT_LAP    = 'NÚMERO DE HOGARES CON TECNOLOGÍAS TIC | 2024 | Computadora o laptop o tablet | Tiene'
CT_PHO    = 'NÚMERO DE HOGARES CON TECNOLOGÍAS TIC | 2024 | Teléfono | Tiene'

# ── TIC columns — No tiene ───────────────────────────────────────────────────
CT_TV_NO  = 'NÚMERO DE HOGARES CON TECNOLOGÍAS TIC | 2024 | Televisor | No tiene'
CT_RAD_NO = 'NÚMERO DE HOGARES CON TECNOLOGÍAS TIC | 2024 | Radio o equipo de sonido | No tiene'
CT_LAP_NO = 'NÚMERO DE HOGARES CON TECNOLOGÍAS TIC | 2024 | Computadora o laptop o tablet | No tiene'
CT_PHO_NO = 'NÚMERO DE HOGARES CON TECNOLOGÍAS TIC | 2024 | Teléfono | No tiene'


def get_penet(muni_tioc, dept):
    mask = (df_census['MUNICIPIO/TIOC'].str.strip() == muni_tioc) & \
           (df_census['DEPARTAMENTO'].str.strip()   == dept)
    if not mask.any():
        raise ValueError(f"CSV_final row not found: '{muni_tioc}' ({dept})")
    r = df_census[mask].iloc[0]

    # Denominator = Tiene + No_tiene (Sin especificar excluded from denominator)
    frg = float(r[CE_FRG]); frg_no = float(r[CE_FRG_NO])
    mcr = float(r[CE_MCR]); mcr_no = float(r[CE_MCR_NO])
    cal = float(r[CE_CAL]); cal_no = float(r[CE_CAL_NO])
    ac  = float(r[CE_AC]);  ac_no  = float(r[CE_AC_NO])
    was = float(r[CE_WAS]); was_no = float(r[CE_WAS_NO])

    tv  = float(r[CT_TV]);  tv_no  = float(r[CT_TV_NO])
    rad = float(r[CT_RAD]); rad_no = float(r[CT_RAD_NO])
    lap = float(r[CT_LAP]); lap_no = float(r[CT_LAP_NO])
    pho = float(r[CT_PHO]); pho_no = float(r[CT_PHO_NO])

    return {
        'total_hh':     float(r[CE_TOT]),
        'refrigerador': frg / (frg + frg_no),
        'microwave':    mcr / (mcr + mcr_no),
        'caleton':      cal / (cal + cal_no),
        'ac':           ac  / (ac  + ac_no),
        'lavadora':     was / (was + was_no),
        'tv':           tv  / (tv  + tv_no),
        'radio':        rad / (rad + rad_no),
        'laptop':       lap / (lap + lap_no),
        'phone':        pho / (pho + pho_no),
        'antenna':      rad / (rad + rad_no),  # same as radio — antenna captures radio/TV broadcast
    }


penet_rows = []
for _, row in df_out.iterrows():
    cn = census_name(row['municipality'])
    d  = get_penet(cn, row['department'])
    d['municipality'] = row['municipality']
    penet_rows.append(d)

df_penet = pd.DataFrame(penet_rows).set_index('municipality')

print("Appliance penetrations — p = Tiene/(Tiene+No_tiene), census 2024:\n")
disp_cols   = ['refrigerador', 'ac', 'caleton', 'lavadora', 'microwave',
                'tv', 'radio', 'phone', 'laptop']
disp_labels = ['Fridge', 'AC', 'Calefon', 'WashMach', 'Microwave',
               'TV', 'Radio', 'Phone', 'Laptop']
df_p = df_penet[disp_cols].copy()
df_p.columns = disp_labels
print(df_p.to_string(float_format='{:.3f}'.format))
print("\nAntenna penetration = Radio penetration (both use CT_RAD Tiene / (Tiene + No_tiene))")
print("Source: output/CSV_final.csv — denominator = Tiene + No_tiene (Sin especificar excluded)")


Appliance penetrations — p = Tiene/(Tiene+No_tiene), census 2024:

                       Fridge    AC  Calefon  WashMach  Microwave    TV  Radio  Phone  Laptop
municipality                                                                                 
Riberalta               0.498 0.069    0.026     0.235      0.074 0.649  0.547  0.868   0.228
Guayaramerín            0.541 0.114    0.025     0.352      0.085 0.664  0.475  0.839   0.226
Reyes                   0.423 0.052    0.012     0.180      0.064 0.490  0.418  0.838   0.147
Santa Rosa              0.425 0.062    0.025     0.168      0.051 0.487  0.317  0.876   0.148
Exaltación              0.099 0.005    0.016     0.022      0.003 0.298  0.485  0.751   0.070
El Sena                 0.214 0.016    0.013     0.076      0.014 0.367  0.355  0.787   0.105
Puerto Gonzalo Moreno   0.278 0.009    0.015     0.057      0.020 0.475  0.522  0.747   0.106
Villa Nueva             0.171 0.003    0.003     0.021      0.006 0.319  0.395  0.743  

### 2.2 Annual kWh per Appliance

**A — From RAMP parameters** (read directly; no simulation needed)

For each appliance: `kWh/day = number × power_W × func_time_min / 60 / 1000 × occasional_use`

Parameters are taken from the T2 section of `inputs/sufficiency_inputs/households/lowlands_household_[season].py` — the Tier 2 sufficiency household model for lowlands Bolivia, which applies to all Norte Amazónica municipalities. Where `func_time` varies across the 4 seasonal files, the **mean of the 4 seasons** is used. The `windows` and `time_fraction_random` arguments only affect *when* during the day the load occurs, not *how much* energy is consumed — they are therefore ignored for annual kWh calculation.

**B — Published values** (MHE-GIZ 2022, via Pablo Gómez EnergyScope Bolivia national model)

| Appliance | kWh/year | End-use |
|---|---|---|
| Refrigerador | 458 | FOOD_PRESERVATION |
| Aire Acondicionado | 451 | SPACE_COOLING |
| Calefón eléctrique | 824 | HEAT_LOW_T_HW |
| Microondas | 50 | ELECTRICITY |

For the water heater, 97.5 % of *calefones* in Norte Amazónica are electric (no natural gas network); the effective penetration used is therefore `p_caleton × 0.975`.

**Fan** is handled separately: its annual kWh per household depends on local temperatures and is read from `data/thermal_comfort_lookup.csv` (one value per municipality and season).


In [11]:

# RAMP_BASE = "../../RAMP_Bolivia/inputs/sufficiency_inputs/households"

# Indoor lighting: 4 × 7 W LED, func_time mean of 4 seasons (fall=400, spring=270, summer=270, winter=400 min/day)
INDOOR_BULB_FT  = (400 + 270 + 270 + 400) / 4          # = 335 min/day
KWH_INDOOR_BULB = 4 * 7 * INDOOR_BULB_FT / 60 / 1000 * 365

# ICT appliances (T2, same parameters across all 4 seasonal files)
KWH_TV      = 1 *  30 * 120 / 60 / 1000 * 365
KWH_RADIO   = 1 *  36 * 120 / 60 / 1000 * 365
KWH_PHONE   = 4 *   5 * 120 / 60 / 1000 * 365
KWH_LAPTOP  = 1 *  70 *  90 / 60 / 1000 * 365

# Blender (occasional_use=0.33) and washing machine (occasional_use=0.15)
KWH_BLENDER = 1 *  50 *  30 / 60 / 1000 * 0.33 * 365
KWH_WASHING = 1 * 800 * 120 / 60 / 1000 * 0.15 * 365

# Antenna (proxy device for Internet/cable subscribers; 8 W, 80 min/day)
KWH_ANTENNA = 1 *   8 *  80 / 60 / 1000 * 365

FAN_POWER_W = 30

# Published kWh/year per appliance (MHE-GIZ 2022 via Pablo Gómez EnergyScope Bolivia)
KWH_FRIDGE        = 458
KWH_AC            = 451
KWH_CALETON       = 824
KWH_MICRO         = 50
FRAC_ELEC_CALETON = 0.975   # 97.5% of water heaters in Norte Amazónica are electric

print(f"Indoor bulb: {KWH_INDOOR_BULB:.2f} kWh/year  (4 × 7 W, {INDOOR_BULB_FT:.0f} min/day mean)")
print(f"Fridge: {KWH_FRIDGE} | AC: {KWH_AC} | Calefón: {KWH_CALETON} | Micro: {KWH_MICRO} kWh/year")

# Fan kWh/year per municipality from thermal_comfort_lookup
RAMP_DATA = "../../RAMP_Bolivia/data"
df_tc = pd.read_csv(os.path.join(RAMP_DATA, "thermal_comfort_lookup.csv"))

FAN_NAME_MAP = {
    'San_Lorenzo':           'San Lorenzo',
    'Sena':                  'El Sena',
    'Santa_Rosa_Pando':      'Santa Rosa del Abuná',
    'Santa_Rosa_Beni':       'Santa Rosa',
    'Puerto_Gonzalo_Moreno': 'Puerto Gonzalo Moreno',
    'Puerto_Rico':           'Puerto Rico',
    'Bella_Flor':            'Bella Flor',
    'Nueva_Esperanza':       'Nueva Esperanza',
    'Santos_Mercado':        'Santos Mercado',
    'Villa_Nueva':           'Villa Nueva',
    'San_Pedro':             'San Pedro',
}
df_tc['municipality'] = df_tc['municipality'].map(lambda x: FAN_NAME_MAP.get(x, x))

df_fan_mean = df_tc.groupby('municipality')['func_time'].mean()
kwh_fan     = (FAN_POWER_W * df_fan_mean / 60 / 1000 * 365).rename('kwh_fan_year_per_HH')

missing = [m for m in df_out['municipality'] if m not in kwh_fan.index]
assert not missing, f"Missing fan data for: {missing}"

print(f"\nFan kWh/year per household (all 21 municipalities):\n")
print(kwh_fan.loc[df_out['municipality'].values].to_string(float_format='{:.1f}'.format))


Indoor bulb: 57.06 kWh/year  (4 × 7 W, 335 min/day mean)
Fridge: 458 | AC: 451 | Calefón: 824 | Micro: 50 kWh/year

Fan kWh/year per household (all 21 municipalities):

municipality
Riberalta               105.4
Guayaramerín            105.4
Reyes                    84.9
Santa Rosa              108.1
Exaltación              104.0
El Sena                 109.5
Puerto Gonzalo Moreno   104.0
Villa Nueva             109.5
Cobija                   94.4
Porvenir                 94.4
Puerto Rico              99.9
Filadelfia               93.1
Bella Flor               98.6
Bolpebra                 95.8
Ixiamas                  57.5
San Lorenzo             102.7
Santa Rosa del Abuná    101.3
Santos Mercado          109.5
Nueva Esperanza          97.2
Ingavi                  102.7
San Pedro                97.2


### 2.3 Bottom-up Consumption and Uniform Calibration

For each municipality:

1. **Raw end-use consumption**: `raw_EU = Σ( kWh_appliance × penetration × HH_grid )` for each appliance in the category.
2. **Bottom-up total**: `BU_total = Σ(raw_EU)` over all five end-uses.
3. **Calibration factor**: `k = residential_AETN_kWh / BU_total`.
4. **Final end-use**: `final_EU = raw_EU × k` — the five end-uses now sum exactly to the AETN residential total.

The calibration factor `k` absorbs all sources of imprecision (appliances not modelled, differences in actual vs. nominal power, sub-municipal variation). It is applied **uniformly** across all end-uses, so the **relative proportions** are those from the bottom-up model; only the **level** is anchored to the AETN measured data.

The column `raw breakdown (%)` below shows the pre-calibration end-use shares, which are the core output of the bottom-up model.


In [12]:

bu_records = []

print(f"  {'Municipality':<25} {'BU_total':>10} {'AETN_res':>10} {'k':>7}  {'raw breakdown (%)':}")
print(f"  {'':25} {'(kWh)':>10} {'(kWh)':>10} {'':>7}  FOOD  COOL  HEAT   LGT  ELEC")
print('  ' + '-' * 88)

for _, row in df_out.iterrows():
    muni    = row['municipality']
    dept    = row['department']
    hh_grid = get_hh(census_name(muni), dept)
    res_kwh = row['residential_MWh'] * 1000

    p = df_penet.loc[muni]

    food  = KWH_FRIDGE * p['refrigerador'] * hh_grid
    cool  = KWH_AC * p['ac'] * hh_grid + kwh_fan[muni] * hh_grid
    heat  = KWH_CALETON * p['caleton'] * FRAC_ELEC_CALETON * hh_grid
    light = KWH_INDOOR_BULB * hh_grid
    elec  = (
        KWH_TV      * p['tv']        * hh_grid +
        KWH_RADIO   * p['radio']     * hh_grid +
        KWH_PHONE   * p['phone']     * hh_grid +
        KWH_LAPTOP  * p['laptop']    * hh_grid +
        KWH_BLENDER * hh_grid +
        KWH_WASHING * p['lavadora']  * hh_grid +
        KWH_ANTENNA * p['antenna']   * hh_grid +
        KWH_MICRO   * p['microwave'] * hh_grid
    )

    bu_total = food + cool + heat + light + elec
    k = res_kwh / bu_total

    pct = lambda x: x / bu_total * 100
    print(f"  {muni:<25} {bu_total:>10.0f} {res_kwh:>10.0f} {k:>7.3f}  "
          f"{pct(food):5.1f} {pct(cool):5.1f} {pct(heat):5.1f} {pct(light):5.1f} {pct(elec):5.1f}")

    bu_records.append({
        'municipality':          muni,
        'department':            dept,
        'hh_grid':               hh_grid,
        'res_aetn_kWh':          res_kwh,
        'bu_total_kWh':          bu_total,
        'calib_factor':          k,
        'FOOD_PRESERVATION_kWh': food  * k,
        'SPACE_COOLING_kWh':     cool  * k,
        'HEAT_LOW_T_HW_kWh':     heat  * k,
        'LIGHTING_R_C_kWh':      light * k,
        'ELECTRICITY_kWh':       elec  * k,
    })

df_eu = pd.DataFrame(bu_records)

print(f"\n  k range: {df_eu['calib_factor'].min():.3f} – {df_eu['calib_factor'].max():.3f}  "
      f"(mean {df_eu['calib_factor'].mean():.3f})")


  Municipality                BU_total   AETN_res       k  raw breakdown (%)
                                 (kWh)      (kWh)          FOOD  COOL  HEAT   LGT  ELEC
  ----------------------------------------------------------------------------------------
  Riberalta                   12064376   31096080   2.578   43.7  26.1   4.0  10.9  15.2
  Guayaramerín                 5364328   18367610   3.424   43.5  27.5   3.5  10.0  15.4
  Reyes                        1016006    2810710   2.766   44.8  25.1   2.2  13.2  14.6
  Santa Rosa                    807933    2594560   3.211   41.7  29.1   4.4  12.2  12.6
  Exaltación                     76423     194340   2.543   17.3  40.6   5.0  21.8  15.3
  El Sena                       366515    2076350   5.665   30.0  35.7   3.3  17.4  13.6
  Puerto Gonzalo Moreno         491678     541830   1.102   35.8  30.5   3.5  16.1  14.1
  Villa Nueva                    78382     279240   3.563   27.4  38.8   0.9  19.9  12.9
  Cobija                      10

In [13]:

END_USES = ['FOOD_PRESERVATION', 'SPACE_COOLING', 'HEAT_LOW_T_HW', 'LIGHTING_R_C', 'ELECTRICITY']

df_eu_mwh = df_eu.set_index('municipality')[[f'{eu}_kWh' for eu in END_USES]].div(1000)
df_eu_mwh.columns = END_USES

print("Residential end-use consumption by municipality (MWh):\n")
print(df_eu_mwh.to_string(float_format='{:.1f}'.format))

# End-uses must sum to residential_MWh from Part 1
check  = df_eu_mwh.sum(axis=1)
ref    = df_out.set_index('municipality')['residential_MWh']
errors = (check - ref).abs()
assert errors.max() < 0.1, f"Calibration error: {errors.idxmax()} off by {errors.max():.2f} MWh"
print(f"\nMax calibration error: {errors.max():.4f} MWh — row sums match AETN residential totals ✓")

totals = df_eu_mwh.sum()
grand  = totals.sum()
print("\nTotal by end-use (21 municipalities):")
for eu, val in totals.items():
    print(f"  {eu:<22}: {val:>8.1f} MWh  ({val/grand*100:.1f}%)")
print(f"  {'TOTAL':<22}: {grand:>8.1f} MWh")


Residential end-use consumption by municipality (MWh):

                       FOOD_PRESERVATION  SPACE_COOLING  HEAT_LOW_T_HW  LIGHTING_R_C  ELECTRICITY
municipality                                                                                     
Riberalta                        13583.1         8127.3         1252.4        3400.6       4732.7
Guayaramerín                      7994.6         5051.7          646.1        1840.5       2834.7
Reyes                             1260.5          706.2           61.3         371.4        411.3
Santa Rosa                        1081.4          755.6          113.6         316.6        327.3
Exaltación                          33.5           79.0            9.8          42.4         29.6
El Sena                            622.1          741.2           67.6         362.1        283.4
Puerto Gonzalo Moreno              194.1          165.2           18.9          87.1         76.5
Villa Nueva                         76.6          108.3       

### 2.4 Quality Control — Comparison with Pablo's National Model

Our weighted end-use shares are compared with proportions from the national EnergyScope Bolivia model (Beni/Pando region). The `PABLO_SHARES` dictionary below should be filled in once those values are available.

**Expected discrepancies and their interpretation** are printed in the cell output below.


In [14]:

# ── 2.4 Quality Control ───────────────────────────────────────────────────────
# Weighted end-use shares across all 21 municipalities
our_shares = (df_eu_mwh[END_USES].sum() / df_eu_mwh[END_USES].sum().sum()) * 100

# Reference from Pablo Gomez national EnergyScope Bolivia model (Beni/Pando).
# Fill these in once available; None = N/A for now.
PABLO_SHARES = {
    'FOOD_PRESERVATION': None,
    'SPACE_COOLING':     None,
    'HEAT_LOW_T_HW':     None,
    'LIGHTING_R_C':      None,
    'ELECTRICITY':       None,
}

print("Quality control -- end-use proportions (weighted mean, 21 municipalities):\n")
print(f"  {'End-use':<22} {'This model (%)':>15} {'Pablo national (%)':>18}")
print('  ' + '-' * 57)
for eu in END_USES:
    pablo = f"{PABLO_SHARES[eu]:.1f}" if PABLO_SHARES[eu] is not None else "N/A"
    print(f"  {eu:<22} {our_shares[eu]:>15.1f} {pablo:>18}")

print("""
Notes on expected discrepancies (census 2024 vs MHE-GIZ 2022):

FOOD_PRESERVATION:
  Our model uses actual refrigerator ownership from census 2024 per municipality.
  Rural Norte Amazonica municipalities have low penetration (10-25 %), which likely
  differs from the national MHE-GIZ 2022 average used in Pablo's model.

HEAT_LOW_T_HW:
  Norte Amazonica is a hot, humid region with very low water heater penetration
  (< 5 % outside Cobija). The census 2024 municipal values are more representative
  than the MHE-GIZ 2022 national survey, which mixes highland and lowland households.

LIGHTING_R_C:
  Our model assumes all grid-connected households use LED bulbs (4 x 7 W, RAMP T2).
  National models may include higher-wattage legacy bulbs.

In all cases, census 2024 (municipal, recent) takes precedence over MHE-GIZ 2022
national estimates -- as confirmed by the methodology choice for this scenario.
""")

# calib_df and ramp_summary kept in memory for traceability (not exported here)
ramp_summary = pd.DataFrame([
    {'appliance': 'Indoor lighting',     'source': 'RAMP',         'kWh_per_HH_year': round(KWH_INDOOR_BULB, 2), 'end_use': 'LIGHTING_R_C',     'penetration': '1.0 (all HH)'},
    {'appliance': 'Refrigerador',        'source': 'MHE-GIZ 2022', 'kWh_per_HH_year': KWH_FRIDGE,               'end_use': 'FOOD_PRESERVATION', 'penetration': 'census Tiene/(Tiene+No_tiene)'},
    {'appliance': 'Aire Acondicionado',  'source': 'MHE-GIZ 2022', 'kWh_per_HH_year': KWH_AC,                   'end_use': 'SPACE_COOLING',     'penetration': 'census Tiene/(Tiene+No_tiene)'},
    {'appliance': 'Fan',                 'source': 'RAMP + thermal_comfort', 'kWh_per_HH_year': 'per-municipality', 'end_use': 'SPACE_COOLING', 'penetration': '1.0 (all HH)'},
    {'appliance': 'Calefon (x0.975)',    'source': 'MHE-GIZ 2022', 'kWh_per_HH_year': KWH_CALETON,              'end_use': 'HEAT_LOW_T_HW',     'penetration': 'census Tiene/(Tiene+No_tiene) x 0.975'},
    {'appliance': 'TV',                  'source': 'RAMP',         'kWh_per_HH_year': round(KWH_TV, 2),          'end_use': 'ELECTRICITY',       'penetration': 'census Tiene/(Tiene+No_tiene)'},
    {'appliance': 'Radio',               'source': 'RAMP',         'kWh_per_HH_year': round(KWH_RADIO, 2),       'end_use': 'ELECTRICITY',       'penetration': 'census Tiene/(Tiene+No_tiene)'},
    {'appliance': 'Phone charger',       'source': 'RAMP',         'kWh_per_HH_year': round(KWH_PHONE, 2),       'end_use': 'ELECTRICITY',       'penetration': 'census Tiene/(Tiene+No_tiene)'},
    {'appliance': 'Laptop',              'source': 'RAMP',         'kWh_per_HH_year': round(KWH_LAPTOP, 2),      'end_use': 'ELECTRICITY',       'penetration': 'census Tiene/(Tiene+No_tiene)'},
    {'appliance': 'Blender',             'source': 'RAMP',         'kWh_per_HH_year': round(KWH_BLENDER, 2),     'end_use': 'ELECTRICITY',       'penetration': '1.0 (all HH)'},
    {'appliance': 'Washing machine',     'source': 'RAMP',         'kWh_per_HH_year': round(KWH_WASHING, 2),     'end_use': 'ELECTRICITY',       'penetration': 'census Tiene/(Tiene+No_tiene)'},
    {'appliance': 'Antenna (~4 kWh/yr)', 'source': 'RAMP',         'kWh_per_HH_year': round(KWH_ANTENNA, 2),     'end_use': 'ELECTRICITY',       'penetration': 'census Radio Tiene/(Tiene+No_tiene)'},
    {'appliance': 'Microwave',           'source': 'MHE-GIZ 2022', 'kWh_per_HH_year': KWH_MICRO,                 'end_use': 'ELECTRICITY',       'penetration': 'census Tiene/(Tiene+No_tiene)'},
])
calib_df = df_eu[['municipality', 'department', 'hh_grid',
                   'bu_total_kWh', 'res_aetn_kWh', 'calib_factor']].copy()
calib_df['fan_kwh_year_per_HH'] = calib_df['municipality'].map(kwh_fan)


Quality control -- end-use proportions (weighted mean, 21 municipalities):

  End-use                 This model (%) Pablo national (%)
  ---------------------------------------------------------
  FOOD_PRESERVATION                 43.8                N/A
  SPACE_COOLING                     26.2                N/A
  HEAT_LOW_T_HW                      3.9                N/A
  LIGHTING_R_C                      10.6                N/A
  ELECTRICITY                       15.6                N/A

Notes on expected discrepancies (census 2024 vs MHE-GIZ 2022):

FOOD_PRESERVATION:
  Our model uses actual refrigerator ownership from census 2024 per municipality.
  Rural Norte Amazonica municipalities have low penetration (10-25 %), which likely
  differs from the national MHE-GIZ 2022 average used in Pablo's model.

HEAT_LOW_T_HW:
  Norte Amazonica is a hot, humid region with very low water heater penetration
  (< 5 % outside Cobija). The census 2024 municipal values are more representative
  t

---

## EnergyScope End-Use Names — Verification

Before assigning end-uses in Parts 3–6, the exact names were verified against the EnergyScope Bolivia model template `EnergyScope_BO_nord_amazonia/Data/2025/02_REF_REGION/Demands.csv`:

| End-use used in this notebook | Status | Description |
|---|---|---|
| `ELECTRICITY` | ✓ confirmed | Residual / unspecified electrical demand |
| `LIGHTING_R_C` | ✓ confirmed | Building (indoor) lighting |
| `LIGHTING_P` | ✓ confirmed | Public (outdoor) lighting |
| `FOOD_PRESERVATION` | ✓ confirmed | Refrigeration / food cooling |
| `SPACE_COOLING` | ✓ confirmed | Space cooling (air conditioning + fans) |
| `HEAT_LOW_T_HW` | ✓ confirmed | Low-temperature heat — hot water |
| `COOKING` | ✓ confirmed | Cooking heat |
| `HEAT_HIGH_T` | ✓ confirmed | High-temperature industrial heat |
| `MECHANICAL_ENERGY_COMM` | ✓ confirmed | Mechanical energy — commercial sector |
| `MECHANICAL_ENERGY_IND` | ✓ confirmed | Mechanical energy — industrial sector |

No fallbacks needed — all requested end-uses exist in the model.

---

## Part 3 — General Sector End-Use Disaggregation

> **UPDATED — sourced shares replacing previous unsourced proxy.** The previous version used shares with no explicit source citation. Replaced with electricity-only useful-energy shares from Peru, Balance Nacional de Energia Util 2016 (MINEM/BID), Tabla 112 (*Comercio y Servicios Total, energia util, columna Electricidad*), renormalised to 100%. The electricity-only column is essential: a tous-vecteurs share would wrongly assign gas/LPG cooking and water-heating fractions to the AETN electricity meter.

The General (commercial/services) AETN total for each municipality is split into EnergyScope end-uses using fixed proportions from **Peru BNEU 2016 (MINEM/BID), Tabla 112** — electricity-only column — used as proxy for Bolivia (same source family as the national EnergyScope Bolivia model).

| End-use | BNEU Tabla 112 raw share | Description |
|---|---|---|
| `ELECTRICITY` | 0.677 | Artefactos diversos (generic unspecified loads) |
| `SPACE_COOLING` | 0.116 | Aire acondicionado + Ventilacion |
| `FOOD_PRESERVATION` | 0.085 | Conservacion de alimentos |
| `COOKING` | 0.042 | Coccion (electric fraction only) |
| `MECHANICAL_ENERGY_COMM` | 0.027 | Fuerza motriz + Bombeo + Movimiento mercancias |
| `LIGHTING_R_C` | 0.030 | Iluminacion |
| `HEAT_LOW_T_HW` | 0.022 | Calentamiento de agua |

Raw sum = 0.999; renormalised to exactly 1.0 in code.

In [15]:

# Source: Peru BNEU 2016, Tabla 112 (Comercio y Servicios Total,
# energia util, columna Electricidad). Electricity-only shares;
# raw sum = 0.999, renormalised to 1.0 below.
_GENERAL_SHARES_RAW = {
    'ELECTRICITY':            0.677,  # Artefactos diversos (generic unspecified loads)
    'SPACE_COOLING':          0.116,  # Aire acondicionado + Ventilacion
    'FOOD_PRESERVATION':      0.085,  # Conservacion de alimentos
    'COOKING':                0.042,  # Coccion (electric fraction only)
    'MECHANICAL_ENERGY_COMM': 0.027,  # Fuerza motriz + Bombeo + Movimiento mercancias
    'LIGHTING_R_C':           0.030,  # Iluminacion
    'HEAT_LOW_T_HW':          0.022,  # Calentamiento de agua
}
_gen_raw_sum = sum(_GENERAL_SHARES_RAW.values())  # 0.999
GENERAL_SHARES = {eu: v / _gen_raw_sum for eu, v in _GENERAL_SHARES_RAW.items()}
assert abs(sum(GENERAL_SHARES.values()) - 1.0) < 1e-9, "General shares must sum to 1"

gen_records = []
for _, row in df_out.iterrows():
    gen_mwh = row['general_MWh']
    rec = {'municipality': row['municipality']}
    for eu, share in GENERAL_SHARES.items():
        rec[f'{eu}_kWh'] = gen_mwh * share * 1000
    gen_records.append(rec)

df_gen = pd.DataFrame(gen_records).set_index('municipality')
df_gen_mwh = df_gen.div(1000).rename(columns={f'{eu}_kWh': eu for eu in GENERAL_SHARES})

print("General sector end-use consumption by municipality (MWh):\n")
print(df_gen_mwh.to_string(float_format='{:.1f}'.format))

# Per-municipality validation: sum of end-uses vs AETN general category total
GEN_EU = list(GENERAL_SHARES.keys())
print("\nPer-municipality validation -- General sector (MWh):")
print(f"  {'Municipality':<30} {'AETN_gen':>10} {'split_sum':>10} {'abs_err':>9}")
print('  ' + '-' * 63)
gen_errors = []
for _, row in df_out.iterrows():
    muni = row['municipality']
    aetn_val = row['general_MWh']
    split_val = df_gen_mwh.loc[muni, GEN_EU].sum()
    err = abs(split_val - aetn_val)
    gen_errors.append(err)
    print(f"  {muni:<30} {aetn_val:>10.2f} {split_val:>10.2f} {err:>9.4f}")
assert max(gen_errors) < 0.1, f"General split error > 0.1 MWh (max={max(gen_errors):.4f})"
print(f"\nAll General split errors < 0.1 MWh  (max = {max(gen_errors):.2e} MWh)")


General sector end-use consumption by municipality (MWh):

                       ELECTRICITY  SPACE_COOLING  FOOD_PRESERVATION  COOKING  MECHANICAL_ENERGY_COMM  LIGHTING_R_C  HEAT_LOW_T_HW
municipality                                                                                                                      
Riberalta                  11718.6         2007.9             1471.3    727.0                   467.4         519.3          380.8
Guayaramerín                5792.6          992.5              727.3    359.4                   231.0         256.7          188.2
Reyes                        634.4          108.7               79.6     39.4                    25.3          28.1           20.6
Santa Rosa                   569.8           97.6               71.5     35.3                    22.7          25.2           18.5
Exaltación                    50.6            8.7                6.3      3.1                     2.0           2.2            1.6
El Sena                 

## Part 4 — Industrial Sector End-Use Disaggregation

> **UPDATED — sourced shares replacing previous unsourced assumptions.** The previous version assigned 85% to `MECHANICAL_ENERGY_IND`, 10% to `HEAT_HIGH_T`, and 5% to `ELECTRICITY` (generic contextual assumption). Replaced with electricity-only MSME logic from the national EnergyScope Bolivia model (Jimenez Zabalaga et al.): on the AETN electricity meter, industrial consumption = mechanical drive + lighting. `HEAT_HIGH_T` is removed because process heat in castana/rice/timber processing in Norte Amazonica is thermal (gas/biomass-fired), **not** on the electricity meter.

Norte Amazonica has no heavy industry. The industrial base is entirely **MPME** (micro, small and medium enterprises) processing agricultural and forestry products: Brazil nuts (*castana*), rice, timber — dominated by electric motor-driven machinery.

| End-use | Share | Source |
|---|---|---|
| `MECHANICAL_ENERGY_IND` | 95% | EnergyScope Bolivia national model, MSME industrial electricity logic (Jimenez Zabalaga et al.) |
| `LIGHTING_R_C` | 5% | Same |

Raw sum = 1.0 exactly; renormalisation is a no-op.

In [16]:

# Source: MSME industrial electricity logic from the national EnergyScope Bolivia model
# (Jimenez Zabalaga et al.). Industrial electricity on the AETN meter = mechanical
# drive + lighting. NO HEAT_HIGH_T (process heat is thermal/gas in Norte Amazonica MPME).
_INDUSTRIAL_SHARES_RAW = {
    'MECHANICAL_ENERGY_IND': 0.95,
    'LIGHTING_R_C':          0.05,
}
_ind_raw_sum = sum(_INDUSTRIAL_SHARES_RAW.values())  # 1.0 exactly
INDUSTRIAL_SHARES = {eu: v / _ind_raw_sum for eu, v in _INDUSTRIAL_SHARES_RAW.items()}
assert abs(sum(INDUSTRIAL_SHARES.values()) - 1.0) < 1e-9, "Industrial shares must sum to 1"

ind_records = []
for _, row in df_out.iterrows():
    ind_mwh = row['industrial_MWh']
    rec = {'municipality': row['municipality']}
    for eu, share in INDUSTRIAL_SHARES.items():
        rec[f'{eu}_kWh'] = ind_mwh * share * 1000
    ind_records.append(rec)

df_ind = pd.DataFrame(ind_records).set_index('municipality')
df_ind_mwh = df_ind.div(1000).rename(columns={f'{eu}_kWh': eu for eu in INDUSTRIAL_SHARES})

print("Industrial sector end-use consumption by municipality (MWh):\n")
print(df_ind_mwh.to_string(float_format='{:.1f}'.format))

# Per-municipality validation: sum of end-uses vs AETN industrial category total
IND_EU = list(INDUSTRIAL_SHARES.keys())
print("\nPer-municipality validation -- Industrial sector (MWh):")
print(f"  {'Municipality':<30} {'AETN_ind':>10} {'split_sum':>10} {'abs_err':>9}")
print('  ' + '-' * 63)
ind_errors = []
for _, row in df_out.iterrows():
    muni = row['municipality']
    aetn_val = row['industrial_MWh']
    split_val = df_ind_mwh.loc[muni, IND_EU].sum()
    err = abs(split_val - aetn_val)
    ind_errors.append(err)
    print(f"  {muni:<30} {aetn_val:>10.2f} {split_val:>10.2f} {err:>9.4f}")
assert max(ind_errors) < 0.1, f"Industrial split error > 0.1 MWh (max={max(ind_errors):.4f})"
print(f"\nAll Industrial split errors < 0.1 MWh  (max = {max(ind_errors):.2e} MWh)")


Industrial sector end-use consumption by municipality (MWh):

                       MECHANICAL_ENERGY_IND  LIGHTING_R_C
municipality                                              
Riberalta                             4935.4         259.8
Guayaramerín                          2117.5         111.4
Reyes                                  225.1          11.8
Santa Rosa                             348.3          18.3
Exaltación                               0.1           0.0
El Sena                                339.9          17.9
Puerto Gonzalo Moreno                    0.0           0.0
Villa Nueva                             37.7           2.0
Cobija                                3668.1         193.1
Porvenir                               333.4          17.5
Puerto Rico                            312.7          16.5
Filadelfia                             305.3          16.1
Bella Flor                              58.9           3.1
Bolpebra                                16.1         

## Part 5 — Public Lighting

The Public Lighting AETN total is assigned entirely to `LIGHTING_P` (public outdoor lighting end-use in EnergyScope).

**Note on outdoor RAMP residential lighting**: the RAMP residential model includes an outdoor lighting component that was kept separate in Part 2 (deliberately excluded from the residential bottom-up split). That outdoor RAMP load is **not added here**, because the AETN Public Lighting category already represents the *measured* electricity sold for street lighting — adding the RAMP outdoor estimate would cause double-counting. The AETN figure is the authoritative measured value and takes precedence.

In [17]:

pl_records = []
for _, row in df_out.iterrows():
    pl_records.append({
        'municipality':  row['municipality'],
        'LIGHTING_P_kWh': row['public_lighting_MWh'] * 1000,
    })

df_pl = pd.DataFrame(pl_records).set_index('municipality')
df_pl_mwh = df_pl.div(1000).rename(columns={'LIGHTING_P_kWh': 'LIGHTING_P'})

print("Public Lighting → LIGHTING_P (MWh):\n")
print(df_pl_mwh.to_string(float_format='{:.1f}'.format))
print(f"\nTotal Public Lighting: {df_out['public_lighting_MWh'].sum():.2f} MWh  ✓")


Public Lighting → LIGHTING_P (MWh):

                       LIGHTING_P
municipality                     
Riberalta                  3987.2
Guayaramerín               1243.1
Reyes                       474.1
Santa Rosa                  224.8
Exaltación                   20.0
El Sena                     153.0
Puerto Gonzalo Moreno       124.6
Villa Nueva                  37.1
Cobija                     2207.7
Porvenir                    200.6
Puerto Rico                 188.2
Filadelfia                  183.7
Bella Flor                   35.5
Bolpebra                      9.7
Ixiamas                     296.2
San Lorenzo                 259.5
Santa Rosa del Abuná         38.9
Santos Mercado               12.9
Nueva Esperanza               4.9
Ingavi                        7.8
San Pedro                     0.1

Total Public Lighting: 9709.50 MWh  ✓


## Part 6 — Other Sector

The "Other" AETN tariff category is a residual that does not map to a specific end-use. It is assigned entirely to `ELECTRICITY` (residual / unspecified electrical demand in EnergyScope).

In [18]:

oth_records = []
for _, row in df_out.iterrows():
    oth_records.append({
        'municipality':    row['municipality'],
        'ELECTRICITY_kWh': row['other_MWh'] * 1000,
    })

df_oth = pd.DataFrame(oth_records).set_index('municipality')
df_oth_mwh = df_oth.div(1000).rename(columns={'ELECTRICITY_kWh': 'ELECTRICITY'})

print("Other sector → ELECTRICITY (MWh):\n")
print(df_oth_mwh.to_string(float_format='{:.1f}'.format))
print(f"\nTotal Other: {df_out['other_MWh'].sum():.2f} MWh  ✓")


Other sector → ELECTRICITY (MWh):

                       ELECTRICITY
municipality                      
Riberalta                    989.3
Guayaramerín                  61.9
Reyes                         31.8
Santa Rosa                   115.9
Exaltación                    22.3
El Sena                       30.2
Puerto Gonzalo Moreno          0.0
Villa Nueva                    1.4
Cobija                       309.6
Porvenir                      28.1
Puerto Rico                   26.4
Filadelfia                    25.8
Bella Flor                     5.0
Bolpebra                       1.4
Ixiamas                       19.9
San Lorenzo                   17.4
Santa Rosa del Abuná          43.3
Santos Mercado                14.3
Nueva Esperanza                5.4
Ingavi                         8.6
San Pedro                      0.1

Total Other: 1758.05 MWh  ✓


---

## Part 7 — Combined End-Use Table (All Sectors, Long Format)

All five AETN tariff categories are assembled into a **tidy (long) format** DataFrame `df_long` with one row per *(municipality, sector, end_use)*. Sectors are **never summed together** — the sector dimension is preserved throughout.

| Column | Content |
|---|---|
| `municipality` | Municipality name |
| `department` | Bolivian department |
| `sector` | EnergyScope sector label (see mapping below) |
| `end_use` | EnergyScope end-use canonical name |
| `MWh` | Final energy consumption (FEC) in MWh |

**Sector mapping** (AETN category → EnergyScope sector):

| AETN category | Sector label | End-uses |
|---|---|---|
| Residential (Part 2) | `HOUSEHOLDS` | FOOD_PRESERVATION, SPACE_COOLING, HEAT_LOW_T_HW, LIGHTING_R_C, ELECTRICITY |
| General (Part 3) | `SERVICES` | ELECTRICITY, SPACE_COOLING, FOOD_PRESERVATION, COOKING, MECHANICAL_ENERGY_COMM, LIGHTING_R_C, HEAT_LOW_T_HW |
| Industrial (Part 4) | `INDUSTRY` | MECHANICAL_ENERGY_IND, LIGHTING_R_C |
| Public Lighting (Part 5) | `PUBLIC_LIGHTING` | LIGHTING_P |
| Other (Part 6) | `SERVICES_OTHER`* | ELECTRICITY |

*TODO: confirm with supervisor whether "Other" (Especial, Agua Potable, Seguridad Ciudadana) maps to SERVICES or another sector.

**Note on FEC → EUD conversion**: applying Layers_in_out efficiencies is a **separate downstream step** and is not performed here.

In [19]:

# Sector definitions: name → (source DataFrame, end-use columns, AETN validation column)
# Sectors are kept strictly separate — df_long contains one row per (muni, sector, end_use).
SECTOR_DEFS = {
    'HOUSEHOLDS':     (df_eu_mwh,  ['FOOD_PRESERVATION', 'SPACE_COOLING', 'HEAT_LOW_T_HW', 'LIGHTING_R_C', 'ELECTRICITY'], 'residential_MWh'),
    'SERVICES':       (df_gen_mwh, list(GENERAL_SHARES.keys()),    'general_MWh'),
    'INDUSTRY':       (df_ind_mwh, list(INDUSTRIAL_SHARES.keys()), 'industrial_MWh'),
    'PUBLIC_LIGHTING':(df_pl_mwh,  ['LIGHTING_P'],                 'public_lighting_MWh'),
    'SERVICES_OTHER': (df_oth_mwh, ['ELECTRICITY'],                'other_MWh'),  # TODO: confirm sector
}

ref = df_out.set_index('municipality')
records = []
for sector, (df_sec, eu_cols, _) in SECTOR_DEFS.items():
    for muni in df_out['municipality']:
        dept = ref.loc[muni, 'department']
        for eu in eu_cols:
            records.append({
                'municipality': muni,
                'department':   dept,
                'sector':       sector,
                'end_use':      eu,
                'MWh':          round(df_sec.loc[muni, eu], 4),
            })

df_long = pd.DataFrame(records)

# Compact summary: one line per sector
print("Source A — FEC by sector (all 21 municipalities combined):\n")
print(f"  {'Sector':<20} {'End-uses':<65} {'Total MWh':>10}")
print('  ' + '-' * 97)
for sector, (df_sec, eu_cols, _) in SECTOR_DEFS.items():
    total = df_sec[eu_cols].values.sum()
    eu_str = ', '.join(eu_cols)
    print(f"  {sector:<20} {eu_str:<65} {total:>10.1f}")

total_all = df_long['MWh'].sum()
print(f"  {'':20} {'':65} {'----------':>10}")
print(f"  {'TOTAL':<20} {'':65} {total_all:>10.1f}")

# Validation: per-municipality, per-sector sum vs AETN category total
print("\nValidation — per-sector sum vs AETN category (max abs error over 21 municipalities):")
all_ok = True
for sector, (df_sec, eu_cols, aetn_col) in SECTOR_DEFS.items():
    errs = (df_sec[eu_cols].sum(axis=1) - ref[aetn_col]).abs()
    ok = errs.max() < 0.1
    print(f"  {sector:<20}: max err = {errs.max():.2e} MWh  [{'OK' if ok else 'FAIL'}]")
    if not ok:
        all_ok = False
assert all_ok, "One or more sector validation errors exceed 0.1 MWh"
print("All sector validation errors < 0.1 MWh")

grand_aetn = df_out['total_MWh'].sum()
print(f"\nGrand total (long table): {total_all:.2f} MWh")
print(f"AETN grand total (Part 1): {grand_aetn:.2f} MWh")


Source A — FEC by sector (all 21 municipalities combined):

  Sector               End-uses                                                           Total MWh
  -------------------------------------------------------------------------------------------------
  HOUSEHOLDS           FOOD_PRESERVATION, SPACE_COOLING, HEAT_LOW_T_HW, LIGHTING_R_C, ELECTRICITY   100153.5
  SERVICES             ELECTRICITY, SPACE_COOLING, FOOD_PRESERVATION, COOKING, MECHANICAL_ENERGY_COMM, LIGHTING_R_C, HEAT_LOW_T_HW    54836.9
  INDUSTRY             MECHANICAL_ENERGY_IND, LIGHTING_R_C                                  13645.1
  PUBLIC_LIGHTING      LIGHTING_P                                                            9709.5
  SERVICES_OTHER       ELECTRICITY                                                           1758.0
                                                                                         ----------
  TOTAL                                                                                  

In [20]:

# Export in long (tidy) format: one row per (municipality, sector, end_use).
# Sectors are kept separate — do NOT sum across sectors before the EnergyScope step.
CSV_OUT = os.path.join(OUTPUT_DIR, 'source_A_all_sectors_end_uses.csv')
df_long.to_csv(CSV_OUT, index=False, encoding='utf-8-sig', float_format='%.4f')
print(f"Saved: {CSV_OUT}")
print(f"Shape: {df_long.shape[0]} rows x {df_long.shape[1]} columns")
print(f"Sectors  : {sorted(df_long['sector'].unique())}")
print(f"End-uses : {sorted(df_long['end_use'].unique())}")
print(f"\nPreview (first 12 rows):")
print(df_long.head(12).to_string(index=False))


Saved: output\source_A_all_sectors_end_uses.csv
Shape: 336 rows x 5 columns
Sectors  : ['HOUSEHOLDS', 'INDUSTRY', 'PUBLIC_LIGHTING', 'SERVICES', 'SERVICES_OTHER']
End-uses : ['COOKING', 'ELECTRICITY', 'FOOD_PRESERVATION', 'HEAT_LOW_T_HW', 'LIGHTING_P', 'LIGHTING_R_C', 'MECHANICAL_ENERGY_COMM', 'MECHANICAL_ENERGY_IND', 'SPACE_COOLING']

Preview (first 12 rows):
municipality department     sector           end_use        MWh
   Riberalta       Beni HOUSEHOLDS FOOD_PRESERVATION 13583.1370
   Riberalta       Beni HOUSEHOLDS     SPACE_COOLING  8127.2936
   Riberalta       Beni HOUSEHOLDS     HEAT_LOW_T_HW  1252.4191
   Riberalta       Beni HOUSEHOLDS      LIGHTING_R_C  3400.5709
   Riberalta       Beni HOUSEHOLDS       ELECTRICITY  4732.6594
Guayaramerín       Beni HOUSEHOLDS FOOD_PRESERVATION  7994.5747
Guayaramerín       Beni HOUSEHOLDS     SPACE_COOLING  5051.7272
Guayaramerín       Beni HOUSEHOLDS     HEAT_LOW_T_HW   646.1115
Guayaramerín       Beni HOUSEHOLDS      LIGHTING_R_C  1840.48